# Utils - QB - PeriodPositionConverter

Ce notebook illustre et vérifie le comportement de la classe `PeriodPositionConverter`
(`tsforecast/utils/position/converter.py`), utilisée pour convertir l'index d'une
série temporelle, d'un DataFrame ou d'un panel entre une position de **début de
période** (`'S'`/`'start'`, ex. `MS` = 1er jour du mois) et une position de
**fin de période** (`'E'`/`'end'`, ex. `ME` = dernier jour du mois). Ce type de
conversion est utile pour harmoniser des indicateurs publiés à des positions
différentes (ex. PIB trimestriel en `QS` vs `QE`) avant de les combiner.

**Périmètre** : seules les méthodes publiques de `PeriodPositionConverter`
(définies dans `converter.py`) sont testées :
- `convert(value, from_unit, to_unit, freq=None, **kwargs)`
- `get_conversion_factor(from_unit, to_unit)`
- `convert_offset(offset_str, to_position)`

Les méthodes privées (`_infer_frequency`, `_convert_datetime_index`,
`_convert_time_series`, `_convert_panel`) ne sont pas testées directement :
elles sont couvertes indirectement via `convert()`, qui les appelle en interne
selon le type de `value` (`DatetimeIndex`, `Series`/`DataFrame` à index simple,
ou `Series`/`DataFrame` à `MultiIndex`).

**Remarque sur le périmètre** : le sous-module `tsforecast/utils/position/`
expose également, via `position/utils.py` (et non `converter.py`), des fonctions
libres (`convert_position`, `convert_offset`, etc.) qui enveloppent
`PeriodPositionConverter`/`PeriodPositionNormalizer`. Conformément à la consigne,
ce notebook se limite strictement à la classe définie dans `converter.py` ; ces
fonctions libres ne sont pas testées ici.

**Jeux de données** : on réutilise `df_timeseries`/`df_panel`, créés dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (indicateurs
macroéconomiques mensuels/trimestriels/annuels, délais de publication,
couverture temporelle et fréquence de publication hétérogènes par pays), comme
données réalistes. `PeriodPositionConverter` exigeant un `DatetimeIndex` (simple
ou dernier niveau d'un `MultiIndex`), ces jeux s'y prêtent directement sans
retraitement. On les complète par de petits jeux synthétiques ciblés pour
couvrir des cas qu'ils ne couvrent pas naturellement : positions déjà explicites
(`QS`/`QE`), panel à plus de deux niveaux d'entité, panel à fréquences mixtes
entre entités, panel mal formé (entités non groupées, dates non triées), et
fréquences aux limites du support de `pandas.Period` (`'B'`, `'W-MON'`, `'SM'`).

**Remarque** : un premier notebook d'exploration plus succinct existait dans
`0 - QB - Utils.ipynb` (section 6) ; il ne couvrait que `convert_offset()` sur
quelques offsets simples. Les classes ayant depuis été refactorées, il a
essentiellement servi à confirmer que `convert_offset()` restait l'entrée la
plus simple à tester en premier.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/utils/position/test_converter.py`, qui n'existe pas encore).

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings
from typing import get_args

import numpy as np
import pandas as pd

# Classe testée
from tsforecast.utils.position.converter import PeriodPositionConverter

# Types exportés (pour lister les positions supportées, à titre indicatif)
from tsforecast.utils.position.normalizer import PositionType, UserPositionType

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

# Instanciation du convertisseur
converter = PeriodPositionConverter()

print("Positions (codes) déclarées dans le type PositionType :")
print(get_args(PositionType))
print()
print("Positions (littéraux) déclarées dans le type UserPositionType :")
print(get_args(UserPositionType))

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (aucune fonction
partagée n'existe entre notebooks dans ce projet) pour obtenir `df_timeseries`
(séries temporelles macroéconomiques) et `df_panel` (panel France/Allemagne/Italie
à couverture et fréquences hétérogènes).

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle
    df['balance_commerciale_annuelle'] = np.nan
    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Simulation des délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Simulation de données historiques limitées
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


df_timeseries = create_timeseries_dataset()
print(f"df_timeseries : {df_timeseries.shape}, {df_timeseries.index.min().date()} -> {df_timeseries.index.max().date()}")
df_timeseries.tail()

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        publication_months = [1] if params['depenses_frequency'] == 'annuelle' else [1, 4, 7, 10]
        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan
        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


df_panel = create_panel_dataset()
print(f"df_panel : {df_panel.shape}, entités : {df_panel.index.get_level_values('country').unique().tolist()}")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country:12s} {dates_country.min().strftime('%Y-%m')} -> {dates_country.max().strftime('%Y-%m')}")
df_panel.loc['France'].tail()

### 2.2 - Jeux de données synthétiques complémentaires

`df_timeseries`/`df_panel` ne couvrent que des positions implicites (`MS`
mensuel côté source) et des panels à un seul niveau d'entité, bien groupés et
triés. Ces jeux synthétiques comblent les cas que `PeriodPositionConverter` doit
aussi gérer : positions déjà explicites (`QS`/`QE`), panel à 3 niveaux
d'index, panel à fréquences mixtes entre entités, panel mal formé, et
fréquences aux limites du support de `pandas.Period`.

In [ ]:
# DatetimeIndex nu (sans Series), position start explicite -> pour tester le
# branchement de convert() dédié aux DatetimeIndex
dates_index_ms = pd.date_range('2023-01-01', periods=6, freq='MS')

# Séries trimestrielles à position explicite S (QS) et E (QE)
dates_qs = pd.date_range('2020-01-01', periods=8, freq='QS')
serie_qs = pd.Series(np.linspace(100, 135, 8), index=dates_qs, name='pib_qs')
dates_qe = pd.date_range('2020-03-31', periods=8, freq='QE')
serie_qe = pd.Series(np.linspace(100, 135, 8), index=dates_qe, name='pib_qe')

# Série à index irrégulier (espacements non constants) : pd.infer_freq() échoue
dates_irregulieres = pd.DatetimeIndex(['2023-01-01', '2023-02-15', '2023-05-01', '2023-05-02'])
serie_irreguliere = pd.Series(range(4), index=dates_irregulieres, name='irregulier')

# DatetimeIndex à un seul point : pd.infer_freq() ne peut rien inférer non plus
index_un_point = pd.DatetimeIndex(['2023-06-01'])

# Panel à 2 entités de fréquences DIFFÉRENTES (trimestrielle vs mensuelle) :
# cas cible pour freq=None, qui doit inférer la fréquence groupe par groupe
entities_mixte = ['A'] * 3 + ['B'] * 4
dates_mixte = (
    list(pd.date_range('2023-01-01', periods=3, freq='QS'))
    + list(pd.date_range('2020-01-01', periods=4, freq='MS'))
)
idx_panel_mixte = pd.MultiIndex.from_arrays([entities_mixte, dates_mixte], names=['entity', 'date'])
panel_freq_mixte = pd.Series(range(7), index=idx_panel_mixte, name='valeur')

# Panel à 3 niveaux d'index (region, entity, date) : pour vérifier le
# regroupement sur "tous les niveaux sauf le dernier" (n_levels > 2)
regions_3lvl = ['EU'] * 4 + ['NA'] * 2
entities_3lvl = ['France', 'France', 'Allemagne', 'Allemagne', 'USA', 'USA']
dates_3lvl = pd.date_range('2023-01-01', periods=2, freq='QS').tolist() * 3
idx_panel_3lvl = pd.MultiIndex.from_arrays(
    [regions_3lvl, entities_3lvl, dates_3lvl], names=['region', 'entity', 'date']
)
panel_3_niveaux = pd.Series(range(6), index=idx_panel_3lvl, name='valeur')

# Panel MAL FORMÉ n°1 : entités non groupées (blocs non contigus)
idx_panel_non_groupe = pd.MultiIndex.from_arrays(
    [['A', 'B', 'A', 'B'], pd.date_range('2023-01-01', periods=4, freq='QS')],
    names=['entity', 'date']
)
panel_non_groupe = pd.Series(range(4), index=idx_panel_non_groupe, name='valeur')

# Panel MAL FORMÉ n°2 : dates non triées au sein d'un groupe
idx_panel_non_trie = pd.MultiIndex.from_arrays(
    [['A', 'A', 'A'], pd.DatetimeIndex(['2023-07-01', '2023-01-01', '2023-04-01'])],
    names=['entity', 'date']
)
panel_non_trie = pd.Series(range(3), index=idx_panel_non_trie, name='valeur')

# Fréquences aux limites du support de pandas.Period (utilisées en section 5.5)
dates_bday = pd.bdate_range('2023-01-02', periods=6, name='date')
serie_bday = pd.Series(range(6), index=dates_bday, name='valeur')

dates_w_mon = pd.date_range('2023-01-02', periods=5, freq='W-MON')
serie_semaine_ancree = pd.Series(range(5), index=dates_w_mon, name='valeur')

dates_semi_mois = pd.date_range('2023-01-01', periods=6, freq='SMS')
serie_semi_mensuelle = pd.Series(range(6), index=dates_semi_mois, name='valeur')

print(f"dates_index_ms       : {len(dates_index_ms)} points, freq={dates_index_ms.freqstr}")
print(f"serie_qs / serie_qe  : {len(serie_qs)} / {len(serie_qe)} points")
print(f"serie_irreguliere    : {len(serie_irreguliere)} points, espacements non constants")
print(f"panel_freq_mixte     : {panel_freq_mixte.shape}, A=trimestriel, B=mensuel")
print(f"panel_3_niveaux      : {panel_3_niveaux.shape}, niveaux={panel_3_niveaux.index.names}")
print(f"panel_non_groupe     : entités {panel_non_groupe.index.get_level_values('entity').tolist()} (A/B entrelacés)")
print(f"panel_non_trie       : dates {[d.date() for d in panel_non_trie.index.get_level_values('date')]}")
print(f"serie_bday           : freq pandas 'B' (jours ouvrés)")
print(f"serie_semaine_ancree : freq pandas 'W-MON' (semaine ancrée sur lundi)")
print(f"serie_semi_mensuelle : freq pandas 'SMS' (semi-mensuel, début de sous-période)")

## 3 - `get_conversion_factor()`

D'après sa docstring, cette méthode ne calcule **aucun facteur multiplicatif
réel** entre positions : contrairement aux `get_conversion_factor()` de
`DurationConverter`/`FrequencyConverter` (qui renvoient un ratio de durées),
celui de `PeriodPositionConverter` renvoie uniquement `1.0` si les positions
sont identiques, `-1.0` si elles diffèrent — un simple indicateur de « shift
nécessaire ou non », indépendant de toute fréquence.

In [ ]:
# Identité vs positions différentes : uniquement 1.0 / -1.0, jamais autre chose
print("get_conversion_factor('S', 'S')         :", converter.get_conversion_factor('S', 'S'))
print("get_conversion_factor('start', 'start') :", converter.get_conversion_factor('start', 'start'))
print("get_conversion_factor('S', 'E')         :", converter.get_conversion_factor('S', 'E'))
print("get_conversion_factor('end', 'start')   :", converter.get_conversion_factor('end', 'start'))

# Formats mixtes : code et littéral interchangeables, y compris mélangés
resultats = {
    "code -> code": converter.get_conversion_factor('S', 'E'),
    "littéral -> littéral": converter.get_conversion_factor('start', 'end'),
    "code -> littéral": converter.get_conversion_factor('S', 'end'),
    "littéral -> code": converter.get_conversion_factor('start', 'E'),
}
for label, valeur in resultats.items():
    print(f"{label:25s} -> {valeur}")
assert len(set(resultats.values())) == 1, "Les 4 formats devraient donner le même résultat"

In [ ]:
# Erreur : position non supportée (from_unit ou to_unit)
for a, b in [('foo', 'E'), ('S', 'foo')]:
    try:
        converter.get_conversion_factor(a, b)
        print(f"get_conversion_factor({a!r}, {b!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"get_conversion_factor({a!r}, {b!r}) -> ValueError : {e}")

## 4 - `convert_offset()`

Convertit un offset pandas (ex. `'MS'`, `'QE'`) vers la position cible en
recombinant sa fréquence de base avec la nouvelle position (via le
`PeriodPositionNormalizer` interne). Trois points de vigilance ressortent des
essais ci-dessous.

In [ ]:
# Cas de base : conversion S <-> E sur des offsets connus (M, Q, Y)
print("convert_offset('MS', 'end')   :", converter.convert_offset('MS', 'end'))
print("convert_offset('QE', 'start') :", converter.convert_offset('QE', 'start'))
print("convert_offset('YS', 'end')   :", converter.convert_offset('YS', 'end'))
print("convert_offset('WE', 'start') :", converter.convert_offset('WE', 'start'))

# Identité : même position demandée -> l'offset est retourné TEL QUEL (sans
# même être recomposé), y compris s'il porte un multiplicateur ('2MS')
assert converter.convert_offset('MS', 'start') == 'MS'
assert converter.convert_offset('2MS', 'start') == '2MS'
print("OK : convert_offset(offset, position_identique) renvoie l'offset inchangé (multiplicateur inclus)")

In [ ]:
# Point de vigilance n°1 : un multiplicateur ('2MS') est PERDU dès qu'un
# changement de position a réellement lieu (le normalizer ne recompose que la
# base de fréquence + la position, sans réinjecter le multiplicateur)
print("convert_offset('2MS', 'end') :", converter.convert_offset('2MS', 'end'), "(attendu naïvement '2ME', le '2' est perdu)")
print("convert_offset('3QE', 'start') :", converter.convert_offset('3QE', 'start'))

In [ ]:
# Point de vigilance n°2 : pour une fréquence non "position-aware" (hors
# M/Q/Y/W/B), convert_offset() est un NO-OP silencieux quelle que soit la
# position demandée -- aucune erreur, aucun changement
print("convert_offset('D', 'end')   :", converter.convert_offset('D', 'end'), "(D n'a pas de notion de position -> inchangé)")
print("convert_offset('D', 'start') :", converter.convert_offset('D', 'start'), "(idem, alors que 'start' != position implicite de 'D')")

# Point de vigilance n°3 : un offset totalement inconnu tombe, via
# decompose_offset(), sur une position par défaut 'E' -- puis
# combine_frequency_position() constate que la fréquence n'est pas
# "position-aware" et renvoie l'offset INCHANGÉ, y compris en demandant 'start'
print("convert_offset('foo', 'end')   :", converter.convert_offset('foo', 'end'), "(inconnu, tombe sur position 'E' par défaut -> demande cohérente, no-op)")
print("convert_offset('foo', 'start') :", converter.convert_offset('foo', 'start'), "(inconnu, position demandée 'start' != 'E' par défaut, mais TOUJOURS no-op, sans erreur)")

## 5 - `convert()` : point d'entrée générique

Dispatché selon le type de `value` : `DatetimeIndex` nu, `Series`/`DataFrame` à
index simple (`DatetimeIndex`), ou `Series`/`DataFrame` à `MultiIndex` (panel,
dernier niveau `DatetimeIndex`).

### 5.1 - `DatetimeIndex` nu

In [ ]:
# freq explicite
r_explicite = converter.convert(dates_index_ms, 'start', 'end', freq='M')
print("freq='M' explicite :", r_explicite)

# freq=None : auto-inférée depuis l'index (pd.infer_freq)
r_auto = converter.convert(dates_index_ms, 'start', 'end')
print("\nfreq=None (auto)   :", r_auto)
assert r_auto.equals(r_explicite)

# Point de vigilance : la conversion S -> E ne tombe PAS sur minuit du dernier
# jour, mais sur 23:59:59.999999999 (to_timestamp(how='end') de pandas) --
# une comparaison stricte avec une date "ronde" (ex. pd.Timestamp('2023-01-31'))
# échouerait silencieusement
print("\nPremière date convertie :", repr(r_auto[0]))
assert r_auto[0] != pd.Timestamp('2023-01-31')
assert r_auto[0] == pd.Timestamp('2023-01-31 23:59:59.999999999')

In [ ]:
# freq=None sur un index sans fréquence détectable -> ValueError explicite
for label, idx in [("irrégulier", dates_irregulieres), ("un seul point", index_un_point)]:
    try:
        converter.convert(idx, 'start', 'end')
        print(f"{label} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{label:14s} -> ValueError : {e}")

### 5.2 - Identité (`from_unit == to_unit`) : aliasing, pas une copie

Quand les positions normalisées sont identiques, `convert()` retourne
**l'objet d'entrée lui-même** (`is`, pas seulement `==`) -- avant même de
regarder le type de `value`. Aucune copie n'est faite : modifier le résultat en
place modifierait aussi l'original.

In [ ]:
serie_identite = pd.Series([1, 2, 3], index=dates_index_ms[:3], name='valeur')
r_identite = converter.convert(serie_identite, 'start', 'start')
print("Series  : même objet retourné ->", r_identite is serie_identite)

df_identite = pd.DataFrame({'a': [1, 2, 3]}, index=dates_index_ms[:3])
r_identite_df = converter.convert(df_identite, 'S', 'S')
print("DataFrame : même objet retourné ->", r_identite_df is df_identite)

r_identite_panel = converter.convert(panel_freq_mixte, 'end', 'end')
print("Panel (MultiIndex) : même objet retourné ->", r_identite_panel is panel_freq_mixte)

### 5.3 - `Series` / `DataFrame` à index simple

In [ ]:
# Series, freq=None auto-inférée (QS détecté)
r_series = converter.convert(serie_qs, 'start', 'end')
print("serie_qs (QS) -> end, auto-inférée :")
print(r_series.head(3))
print(f"Type conservé : {type(r_series).__name__}, name={r_series.name!r}")

# Cohérence : partir de QE et redemander 'start' doit redonner (à la position
# près) le même calendrier que convertir serie_qs vers 'end'
r_qe_to_start = converter.convert(serie_qe, 'end', 'start')
print("\nserie_qe (QE) -> start, auto-inférée :")
print(r_qe_to_start.head(3))

In [ ]:
# DataFrame : toutes les colonnes partagent le même nouvel index, valeurs
# et noms de colonnes inchangés
df_multi_colonnes = pd.DataFrame(
    {'a': range(6), 'b': range(6, 12)}, index=dates_index_ms
)
r_df = converter.convert(df_multi_colonnes, 'start', 'end')
print(r_df)
assert list(r_df.columns) == list(df_multi_colonnes.columns)
assert (r_df.values == df_multi_colonnes.values).all()

In [ ]:
# freq=None sur une Series/DataFrame dont l'index est irrégulier -> ValueError
try:
    converter.convert(serie_irreguliere, 'start', 'end')
except ValueError as e:
    print("Series à index irrégulier, freq=None -> ValueError :", e)

# ... mais freq explicite contourne l'inférence (résultat correct si la
# fréquence fournie est cohérente avec les VRAIS espacements ; ici elle ne
# l'est pas complètement, mais aucune vérification n'est faite : la conversion
# a lieu quand même, silencieusement)
r_force = converter.convert(serie_irreguliere, 'start', 'end', freq='D')
print("\nfreq='D' forcée malgré l'irrégularité (aucune vérification) :")
print(r_force)

### 5.4 - Panel (`MultiIndex`) : fréquence par entité vs fréquence imposée

Le commentaire du code est explicite : *« pour les panels, l'inférence se fait
par groupe »*. Avec `freq=None`, chaque entité est convertie avec **sa propre**
fréquence, détectée indépendamment -- ce qui permet de gérer un panel à
fréquences mixtes (`panel_freq_mixte` : entité A trimestrielle, entité B
mensuelle) en un seul appel.

In [ ]:
# freq=None : chaque entité est convertie avec sa propre fréquence détectée
r_panel_auto = converter.convert(panel_freq_mixte, 'start', 'end')
print(r_panel_auto)

**Point de vigilance majeur** : passer un `freq` explicite à un panel
l'applique **uniformément à toutes les entités**, sans jamais vérifier qu'il
correspond à la fréquence réelle de chacune. Sur `panel_freq_mixte` (A =
trimestriel, B = mensuel), forcer `freq='Q'` traite family B comme si ses dates
mensuelles étaient des débuts de trimestre : plusieurs dates mensuelles
distinctes retombent alors sur le **même** trimestre, produisant des
**doublons d'index** dans le résultat, silencieusement (aucune erreur levée).

In [ ]:
r_panel_force_q = converter.convert(panel_freq_mixte, 'start', 'end', freq='Q')
print(r_panel_force_q)

# Les 4 dates mensuelles de l'entité B (freq='Q' imposée à tort) ne produisent
# plus que 3 dates distinctes après conversion : 2 mois retombent sur le même
# trimestre -> DOUBLON D'INDEX au sein de l'entité B
dates_b_converties = r_panel_force_q.loc['B'].index
print(f"\nEntité B : {len(dates_b_converties)} lignes, {dates_b_converties.nunique()} dates UNIQUES -> doublon(s) : {dates_b_converties.duplicated().any()}")

In [ ]:
# Panel à 3 niveaux d'index (region, entity, date) : le regroupement se fait
# sur TOUS les niveaux sauf le dernier (temps), ici (region, entity) ensemble
r_panel_3lvl = converter.convert(panel_3_niveaux, 'start', 'end', freq='Q')
print(r_panel_3lvl)
assert r_panel_3lvl.index.names == panel_3_niveaux.index.names

In [ ]:
# Panel mal formé n°1 : entités non groupées (blocs non contigus) -> ValueError
try:
    converter.convert(panel_non_groupe, 'start', 'end', freq='Q')
except ValueError as e:
    print("Entités non groupées -> ValueError :", e)

# Panel mal formé n°2 : dates non triées au sein d'un groupe -> ValueError
try:
    converter.convert(panel_non_trie, 'start', 'end', freq='Q')
except ValueError as e:
    print("Dates non triées dans un groupe -> ValueError :", e)

In [ ]:
# MultiIndex dont le dernier niveau n'est PAS un DatetimeIndex -> ValueError
idx_dernier_niveau_invalide = pd.MultiIndex.from_arrays(
    [['A', 'A'], [1, 2]], names=['entity', 'periode_entiere']
)
serie_invalide = pd.Series([1, 2], index=idx_dernier_niveau_invalide)
try:
    converter.convert(serie_invalide, 'start', 'end')
except ValueError as e:
    print("Dernier niveau de MultiIndex non-datetime -> ValueError :", e)

### 5.5 - Types et structures non supportés

In [ ]:
# Type non supporté (ni DatetimeIndex, ni Series/DataFrame)
try:
    converter.convert([1, 2, 3], 'start', 'end')
except ValueError as e:
    print("list -> ValueError :", e)

# Series à RangeIndex (ni DatetimeIndex, ni MultiIndex)
try:
    converter.convert(pd.Series([1, 2, 3]), 'start', 'end')
except ValueError as e:
    print("Series à RangeIndex -> ValueError :", e)

### 5.6 - Fréquences aux limites du support de `pandas.Period`

`_convert_datetime_index()` passe par `DatetimeIndex.to_period(period_freq)`,
qui n'accepte qu'un sous-ensemble des fréquences pandas. `PeriodPositionConverter`
extrait `period_freq` par une regex (`^([DWMQY])`) sur la base de fréquence
décomposée -- ce qui fonctionne pour les bases usuelles mais pas pour toutes.

In [ ]:
# 'B' (jour ouvré) : fonctionne, mais to_period('B') est marqué comme déprécié
# par pandas -- risque de rupture dans une future version de pandas
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    r_bday = converter.convert(serie_bday, 'start', 'end', freq='B')
    print(r_bday)
    if w:
        print(f"\n{len(w)} warning(s) émis, ex. : {w[0].category.__name__} : {w[0].message}")

In [ ]:
# 'W-MON' (semaine ancrée sur lundi) : fonctionne sans souci, la regex
# `^([DWMQY])` matche bien le 'W' initial malgré l'ancre
r_semaine_ancree = converter.convert(serie_semaine_ancree, 'start', 'end', freq='W-MON')
print(r_semaine_ancree)

In [ ]:
# 'SM' (semi-mensuel) : la regex `^([DWMQY])` NE matche PAS ('S' n'est pas
# dans le groupe), period_freq retombe donc sur 'SM' tel quel -- et
# to_period('SM') n'est simplement pas supporté par pandas -> ValueError,
# mais avec un message pandas assez cryptique, pas un message dédié du projet
try:
    converter.convert(serie_semi_mensuelle, 'start', 'end', freq='SM')
except ValueError as e:
    print("freq='SM' -> ValueError (message pandas natif, pas un message projet) :", e)

### 5.7 - Application sur les jeux de données réalistes (`df_timeseries` / `df_panel`)

In [ ]:
# PIB trimestriel (implicitement en position début de mois côté source, freq='MS'
# sur l'index global) : conversion vers 'end' avec freq='Q' explicite (la
# fréquence RÉELLE de la variable, différente de l'index MS du DataFrame)
pib_qs = df_timeseries['pib_trimestriel'].dropna()
pib_qe = converter.convert(pib_qs, 'start', 'end', freq='Q')
print("PIB trimestriel, start -> end :")
print(pib_qe.head())

In [ ]:
# Panel réaliste : chaque pays a sa propre couverture temporelle (début/fin
# différents) -- freq=None infère la fréquence mensuelle par pays indépendamment
prod_ind = df_panel['production_industrielle'].dropna()
prod_ind_end = converter.convert(prod_ind, 'start', 'end')
print("France (début de couverture) :")
print(prod_ind_end.loc['France'].head(3))
print("\nAllemagne (début de couverture différent) :")
print(prod_ind_end.loc['Allemagne'].head(3))

## 6 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/position/test_converter.py` :

- **`get_conversion_factor()`** ne calcule **aucun ratio réel** entre
  positions : `1.0` si identiques, `-1.0` sinon, quelle que soit la fréquence
  (contrairement à `DurationConverter`/`FrequencyConverter`). Accepte codes et
  littéraux indifféremment, y compris mélangés. Lève `ValueError` sur une
  position non supportée.
- **`convert_offset()`** :
  - identité (position cible == position source) : renvoie l'offset **tel
    quel**, multiplicateur inclus (`'2MS'` -> `'2MS'`).
  - changement de position réel : le multiplicateur est **perdu**
    (`'2MS'` -> `'ME'`, pas `'2ME'`).
  - fréquence non "position-aware" (hors `M`/`Q`/`Y`/`W`/`B`, ex. `'D'`) :
    **no-op silencieux** quelle que soit la position demandée, aucune erreur.
  - offset inconnu : `decompose_offset()` retombe sur la position `'E'` par
    défaut, puis `combine_frequency_position()` fait aussi un no-op silencieux
    (la fréquence inconnue n'étant pas "position-aware") -- même en demandant
    `'start'`.
- **`convert()` — identité (`from_unit == to_unit`)** : retourne **l'objet
  d'entrée lui-même** (`is`), sans copie, quel que soit le type (`DatetimeIndex`,
  `Series`, `DataFrame`, panel `MultiIndex`). Point de vigilance pour du code
  appelant qui muterait le résultat en place.
- **`convert()` — position `'end'`** : les timestamps produits par
  `to_timestamp(how='end')` tombent sur `23:59:59.999999999` du dernier jour de
  la période, **pas** sur minuit -- une comparaison avec une date "ronde"
  (`pd.Timestamp('2023-01-31')`) échoue silencieusement.
- **`freq=None`** : auto-inféré via `pd.infer_freq()` (nécessite un index
  régulier d'au moins 3 points) ; `ValueError` explicite sinon (index
  irrégulier, ou à un seul point). Un `freq` explicite contourne totalement
  cette vérification, y compris si l'index réel ne le respecte pas.
- **Panel (`MultiIndex`)** :
  - `freq=None` : fréquence inférée **indépendamment pour chaque entité**
    (regroupement sur tous les niveaux sauf le dernier) -- gère nativement les
    panels à fréquences mixtes.
  - `freq` explicite : appliqué **uniformément à toutes les entités**, sans
    aucune vérification de cohérence avec la fréquence réelle de chacune. Sur
    un panel à fréquences mixtes, cela peut produire des **doublons
    d'index** au sein d'une entité, silencieusement (pas d'exception).
  - Validation stricte de la structure : entités non groupées (blocs non
    contigus) et dates non triées au sein d'un groupe lèvent chacune une
    `ValueError` avec un message explicite dédié.
  - Dernier niveau du `MultiIndex` non-`DatetimeIndex` : `ValueError` dédiée.
- **Types/structures non supportés** : `list` (ou tout ce qui n'est ni
  `DatetimeIndex` ni `Series`/`DataFrame`) et `Series`/`DataFrame` à
  `RangeIndex` (ni `DatetimeIndex` ni `MultiIndex`) lèvent toutes deux
  `ValueError`, avec des messages différents.
- **Limites de `pandas.Period`** : la regex interne (`^([DWMQY])`) qui extrait
  la base de fréquence pour `to_period()` fonctionne pour les bases usuelles
  (y compris ancrées, ex. `'W-MON'`) mais échoue pour `'SM'` (semi-mensuel,
  base `'S'` non reconnue) -- `ValueError` levée, mais avec un message pandas
  natif peu explicite plutôt qu'un message dédié au projet. `'B'` (jour ouvré)
  fonctionne mais s'appuie sur `to_period('B')`, une API pandas dépréciée
  (`FutureWarning`) -- risque de rupture dans une future version de pandas.